# Kumbh-Reunite — Inference (missing-person search)

Build a gallery + simulated camera network, then search a missing-person
image and read back the ranked matches with **camera location + time**.

> Research/benchmark use on authorized, public data only. Matches are
> candidates for a human to verify.

In [ ]:
import os, sys
REPO = os.path.abspath('../..')
sys.path.insert(0, REPO)
from face_retrieval.config import load_config, get_logger, set_seed
from face_retrieval.pipeline import SearchPipeline, collect_samples

cfg = load_config()
set_seed(int(cfg.project.seed))
log = get_logger('kumbh', cfg.logging.level)
SAMPLE = os.path.join(REPO, 'backend', '_testdata', 'faces')  # or use a dataset name
cfg.detector.backend, cfg.embedding.backend

In [ ]:
# Build the gallery (auto-detects faces, embeds, FAISS index, simulates cameras)
samples = collect_samples(cfg, 'sample', SAMPLE, limit=None)
pipe = SearchPipeline(cfg, log)
pipe.build_gallery(samples)
print('gallery faces:', pipe.index.size, '| identities:', len(set(pipe.gallery_ids)))

In [ ]:
# Search a missing-person photo
query = pipe.gallery_paths[0]   # swap for any external face image path
res = pipe.search_image(query, top_k=5)
for m in res['matches']:
    o = m['observation'] or {}
    print(f"#{m['rank']} id={m['identity']:<10} score={m['score']:.3f} "
          f"seen: cam {o.get('camera_id')} @ ({o.get('lat')},{o.get('lng')}) {o.get('timestamp','')}")

In [ ]:
# Visualise the query + top-k montage
from face_retrieval.modules import visualization as viz
from IPython.display import Image
rows = [{'path': m['gallery_path'], 'score': m['score'],
         'correct': m['identity'] == res['matches'][0]['identity']} for m in res['matches']]
out = viz.plot_retrieval_example(query, rows, os.path.join(cfg.paths.output_dir, 'nb_search.png'))
Image(out)